In [1]:
records = [
    {
        "patient_id": "P001",
        "admission_date": "2024-01-10",
        "discharge_date": "2024-01-15",
        "diagnoses": ["E11", "I10"],
        "procedures": ["INSULIN_START"],
        "notes": "Patient diabete type 2 desequilibre. Mise sous insuline."
    },
    {
        "patient_id": "P002",
        "admission_date": "2024-02-03",
        "discharge_date": "2024-02-05",
        "diagnoses": ["J18"],
        "procedures": [],
        "notes": "Pneumonie lobaire droite. Antibiotherapie."
    },
    {
        "patient_id": "P001",
        "admission_date": "2024-03-01",
        "discharge_date": "2024-03-20",
        "diagnoses": ["E11"],
        "procedures": ["INSULIN_ADJUST"],
        "notes": "Rehospitalisation pour diabete mal controle."
    },
    {
        "patient_id": "P003",
        "admission_date": "2024-01-12",
        "discharge_date": "2024-01-13",
        "diagnoses": [],
        "procedures": [],
        "notes": "Douleur thoracique atypique. Bilan negatif."
    },
    {
        "patient_id": "P004",
        "admission_date": "2024-04-01",
        "discharge_date": None,
        "diagnoses": ["I21"],
        "procedures": ["ANGIOPLASTY"],
        "notes": "Infarctus aigu du myocarde. Angioplastie en urgence."
    },
]


In [21]:
from datetime import datetime, timedelta
from collections import Counter

TODAY = datetime(2024, 6, 1)


def clean_records(records):
    anomalies = []

    for record in records:
        try:
            admission = datetime.strptime(record["admission_date"], "%Y-%m-%d")
        except Exception:
            anomalies.append(record)
            continue

        discharge_raw = record["discharge_date"]

        # diagnostic vide
        if not record["diagnoses"]:
            anomalies.append(record)

        # hospitalisation trop longue sans sortie
        if discharge_raw is None:
            if TODAY - admission > timedelta(days=60):
                anomalies.append(record)
            continue

        try:
            discharge = datetime.strptime(discharge_raw, "%Y-%m-%d")
        except Exception:
            anomalies.append(record)
            continue

        if discharge < admission:
            anomalies.append(record)

    return anomalies


def synthetic_report(records):
    report = {}

    durations = []
    patients = {}
    diag_counter = Counter()
    diabetes_count = 0

    for record in records:
        pid = record["patient_id"]
        admission = datetime.strptime(record["admission_date"], "%Y-%m-%d")
        discharge_raw = record["discharge_date"]

        # init patient
        if pid not in patients:
            patients[pid] = {
                "count": 0,
                "total_days": 0,
                "admissions": []
            }

        patients[pid]["count"] += 1

        # durée
        if discharge_raw is not None:
            discharge = datetime.strptime(discharge_raw, "%Y-%m-%d")
            days = (discharge - admission).days
            durations.append(days)
            patients[pid]["total_days"] += days
            patients[pid]["admissions"].append((admission, discharge))

        # diagnostics
        diag_counter.update(record["diagnoses"])

        # diabete
        notes = record["notes"].lower()
        if "diabete" in notes or "diabetes" in notes:
            diabetes_count += 1

    # durée moyenne
    report["average_duration"] = (
        sum(durations) / len(durations) if durations else 0
    )

    # nombre patients
    report["total_patients"] = len(patients)

    # readmission ≤ 30 jours
    for pid, data in patients.items():
        admissions = sorted(data["admissions"], key=lambda x: x[0])
        readmit = False

        for i in range(1, len(admissions)):
            prev_discharge = admissions[i - 1][1]
            curr_admission = admissions[i][0]
            if (curr_admission - prev_discharge).days <= 30:
                readmit = True
                break

        data["readmitted_30_days"] = readmit
        del data["admissions"]

    report["patients"] = patients
    report["top_3_diagnostics"] = diag_counter.most_common(3)
    report["diabetes_hospitalizations"] = diabetes_count
    report["anomalies"] = clean_records(records)

    return report

In [30]:
records = [
    {
        "patient_id": "P1",
        "diagnosis": "E11",
        "cost": 1200
    },
    {
        "patient_id": "P2",
        "diagnosis": "I10",
        "cost": 800
    },
    {
        "patient_id": "P1",
        "diagnosis": "E11",
        "cost": 600
    },
    {
        "patient_id": "P3",
        "diagnosis": "J18",
        "cost": 1500
    },
    {
        "patient_id": "P2",
        "diagnosis": "E11",
        "cost": 400
    }
]

from collections import Counter

def valid_record(record: dict) -> bool:
    for key, val in record.items():
        if val is None:
            return False
    else:
        return True

def synthetic_report(records: list[dict]) -> dict:
    patient_dict = {}
    max_cost = 0
    max_costed_patient = ""
    diagnosis = []
    
    for record in records:
        if not valid_record(record):
            continue

        patient_id = record["patient_id"]
        cost = record["cost"]
        diagnosis.append(record["diagnosis"])
        
        if patient_id not in patient_dict:
           patient_dict[patient_id] = {
                "cost": 0,
                "nb_hospi": 0
            }
        patient_dict[patient_id]["cost"] += cost
        patient_dict[patient_id]["nb_hospi"] += 1

        # max cost
        if max_cost < patient_dict[patient_id]["cost"]:
            max_cost = patient_dict[patient_id]["cost"]
            max_costed_patient = patient_id

    # Most common diagnosis
    diag_counter = Counter(diagnosis)
    most_common = diag_counter.most_common(1)

    if most_common:
        most_common_diag = most_common[0][0]
    else:
        most_common_diag = None
        
    return {
        "patient_info": patient_dict,
        "max_cost": max_cost,
        "max_costed_patient": max_costed_patient,
        "most_common_diag": most_common_diag
    }

synthetic_report(records)

{'patient_info': {'P1': {'cost': 1800, 'nb_hospi': 2},
  'P2': {'cost': 1200, 'nb_hospi': 2},
  'P3': {'cost': 1500, 'nb_hospi': 1}},
 'max_cost': 1800,
 'max_costed_patient': 'P1',
 'most_common_diag': 'E11'}